In [1]:
import pandas
import pandas as pd
import re
import unicodedata
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from tqdm import tqdm
import time


In [4]:
df = pd.read_csv("combined_tafseer.csv")
df

,Surah Number,Ayah Number,Text
0,1,1,Introduction to Fatihah\nWhich was revealed in...
1,1,2,The Meaning of Al-Hamd\nAbu Ja`far bin Jarir s...
2,1,3,"Allah said next,\nالرَّحْمَـنِ الرَّحِيمِ\n(Ar..."
3,1,4,Indicating Sovereignty on the Day of Judgment\...
4,1,5,The Linguistic and Religious Meaning of `Ibada...
...,...,...,...
6231,114,2,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...
6232,114,3,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...
6233,114,4,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...
6234,114,5,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...


In [9]:
df.iloc[0]["Text"]

'Introduction to Fatihah\nWhich was revealed in Makkah\nThe Meaning of Al-Fatihah and its Various Names\nThis Surah is called\n- Al-Fatihah, that is, the Opener of the Book, the Surah with which prayers are begun.\n- It is also called, Umm Al-Kitab (the Mother of the Book), according to the majority of the scholars.\nIn an authentic Hadith recorded by At-Tirmidhi, who graded it Sahih, Abu Hurayrah said that the Messenger of Allah ﷺ said,\nالْحَمْدُ للهِ رَبَ الْعَالَمِينَ أُمُّ الْقُرْآنِ وَأُمُّ الْكِتَابِ وَالسَّبْعُ الْمَثَانِي وَالْقُرْآنُ الْعَظِيمُ\nAl-Hamdu lillahi Rabbil-`Alamin is the Mother of the Qur\'an, the Mother of the Book, and the seven repeated Ayat of the Glorious Qur\'an.It is also called Al-Hamd and As-Salah, because the Prophet ﷺ said that his Lord said,\nقَسَمْتُ الصَّلَاةَ بَيْنِي وَبَيْنَ عَبْدِي نِصْفَيْنِ، فَإِذَا قَالَ الْعَبْدُ:الْحَمْدُ للهِ رَبِّ الْعَالَمِينَ، قَالَ اللهُ: حَمِدَنِي عَبْدِي\nThe prayer (i.e., Al-Fatihah) is divided into two halves betwee

In [5]:
ARABIC_TASHKEEL = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0657-\u065F\u0670]')



def preprocess_tafsir_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text(separator=" \n ")

    text = unicodedata.normalize("NFKC", text)

    text = re.sub(ARABIC_TASHKEEL, '', text)

    text = text.replace('«', '(').replace('»', ')').replace('ـ', '')

    text = re.sub('[إأآٱ]', 'ا', text)
    text = re.sub('ى', 'ي', text)
    text = re.sub('ؤ', 'و', text)
    text = re.sub('ئ', 'ي', text)
    text = re.sub('ة', 'ه', text)
    text = re.sub('ﷺ', '', text)

    text = re.sub(
        r'\((\d+):(\d+)(?:-(\d+))?\)',
        r'(سوره \1 الايه \2\3)',
        text
    )
    lines = []
    for line in text.splitlines():
        line = re.sub(r'\s+', ' ', line).strip()
        if line:
            lines.append(line)

    text = "\n".join(lines)
    text = re.sub(r'\n{2,}', '\n', text).strip()

    return text


In [6]:
preprocess_tafsir_text(df['Text'][0])

'Introduction to Fatihah\nWhich was revealed in Makkah\nThe Meaning of Al-Fatihah and its Various Names\nThis Surah is called\n- Al-Fatihah, that is, the Opener of the Book, the Surah with which prayers are begun.\n- It is also called, Umm Al-Kitab (the Mother of the Book), according to the majority of the scholars.\nIn an authentic Hadith recorded by At-Tirmidhi, who graded it Sahih, Abu Hurayrah said that the Messenger of Allah صلي الله عليه وسلم said,\nالحمد لله رب العالمين ام القران وام الكتاب والسبع المثاني والقران العظيم\nAl-Hamdu lillahi Rabbil-`Alamin is the Mother of the Qur\'an, the Mother of the Book, and the seven repeated Ayat of the Glorious Qur\'an.It is also called Al-Hamd and As-Salah, because the Prophet صلي الله عليه وسلم said that his Lord said,\nقسمت الصلاه بيني وبين عبدي نصفين، فاذا قال العبد:الحمد لله رب العالمين، قال الله: حمدني عبدي\nThe prayer (i.e., Al-Fatihah) is divided into two halves between Me and My servants. When the servant says, `All praise is due to

In [10]:
df["tafseer"] = df["Text"].apply(preprocess_tafsir_text)

In [11]:
df

,Surah Number,Ayah Number,Text,tafseer
0,1,1,Introduction to Fatihah\nWhich was revealed in...,Introduction to Fatihah\nWhich was revealed in...
1,1,2,The Meaning of Al-Hamd\nAbu Ja`far bin Jarir s...,The Meaning of Al-Hamd\nAbu Ja`far bin Jarir s...
2,1,3,"Allah said next,\nالرَّحْمَـنِ الرَّحِيمِ\n(Ar...","Allah said next,\nالرحمن الرحيم\n(Ar-Rahman (t..."
3,1,4,Indicating Sovereignty on the Day of Judgment\...,Indicating Sovereignty on the Day of Judgment\...
4,1,5,The Linguistic and Religious Meaning of `Ibada...,The Linguistic and Religious Meaning of `Ibada...
...,...,...,...,...
6231,114,2,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...,Which was revealed in Makkah\nبسم الله الرحمن ...
6232,114,3,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...,Which was revealed in Makkah\nبسم الله الرحمن ...
6233,114,4,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...,Which was revealed in Makkah\nبسم الله الرحمن ...
6234,114,5,Which was revealed in Makkah\nبِسْمِ اللَّهِ ا...,Which was revealed in Makkah\nبسم الله الرحمن ...


In [12]:
df.drop('Text', axis=1, inplace=True)

In [13]:
df

,Surah Number,Ayah Number,tafseer
0,1,1,Introduction to Fatihah\nWhich was revealed in...
1,1,2,The Meaning of Al-Hamd\nAbu Ja`far bin Jarir s...
2,1,3,"Allah said next,\nالرحمن الرحيم\n(Ar-Rahman (t..."
3,1,4,Indicating Sovereignty on the Day of Judgment\...
4,1,5,The Linguistic and Religious Meaning of `Ibada...
...,...,...,...
6231,114,2,Which was revealed in Makkah\nبسم الله الرحمن ...
6232,114,3,Which was revealed in Makkah\nبسم الله الرحمن ...
6233,114,4,Which was revealed in Makkah\nبسم الله الرحمن ...
6234,114,5,Which was revealed in Makkah\nبسم الله الرحمن ...


In [17]:
df = df.sort_values(["Surah Number", "Ayah Number"]).reset_index(drop=True)

df['group'] = (df['tafseer'] != df['tafseer'].shift()) | (df['Surah Number'] != df['Surah Number'].shift())
df['group_id'] = df['group'].cumsum()

result = df.groupby(['Surah Number','group_id']).agg(
    surah_no=('Surah Number', 'first'),
    ayah_start=('Ayah Number', 'min'),
    ayah_end=('Ayah Number', 'max'),
    tafseer=('tafseer', 'first')
).reset_index(drop=True)

result


,surah_no,ayah_start,ayah_end,tafseer
0,1,1,1,Introduction to Fatihah\nWhich was revealed in...
1,1,2,2,The Meaning of Al-Hamd\nAbu Ja`far bin Jarir s...
2,1,3,3,"Allah said next,\nالرحمن الرحيم\n(Ar-Rahman (t..."
3,1,4,4,Indicating Sovereignty on the Day of Judgment\...
4,1,5,5,The Linguistic and Religious Meaning of `Ibada...
...,...,...,...,...
1891,110,1,3,The Virtues of Surat An-Nasr\nIt has been ment...
1892,111,1,5,Which was revealed in Makkah\nبسم الله الرحمن ...
1893,112,1,4,Which was revealed in Makkah\nThe Reason for t...
1894,113,1,5,Which was revealed in Makkah\nThe Position of ...


In [20]:
duplicates = result[result.duplicated(subset=['tafseer'], keep=False)]
duplicates

,surah_no,ayah_start,ayah_end,tafseer
1885,104,1,9,Which was revealed in Makkah\nبسم الله الرحمن ...
1886,105,1,5,Which was revealed in Makkah\nبسم الله الرحمن ...


In [25]:
result.loc[result['surah_no'] == 105, 'tafseer'] = f"""
This [event] is one of the blessings Allah granted to the Quraysh: He protected them from the people of the Elephant (the army of Abraha) who had planned to destroy the Kaaba and erase it from existence. Allah destroyed them, humiliated them, ruined their plans, and made their efforts fail, returning them defeated. They were Christians at the time, whose religion was, in some ways, closer to monotheism than the Quraysh’s idolatry.

However, this incident also served as a preparation and prelude for the coming of the Prophet Muhammad (peace be upon him). That year, according to most accounts, he was born. It is as if destiny itself was saying: “We did not grant you victory over the Abyssinians because of your superiority, O Quraysh, but rather to protect the Ancient House [the Kaaba] that we will honor through the mission of the Prophet Muhammad (peace and blessings be upon him), the last of the prophets.”

The Story of the People of the Elephant (Summary)

Previously, in the story of Ashab al-Akhduud (People of the Ditch), we learn that Dhu Nuwas, the last king of Himyar and a polytheist, killed nearly twenty thousand Christians. Only Daws Dhuth Thalban escaped. He sought help from Caesar of Syria, a Christian, who wrote to the Negus (king) of Abyssinia. The Negus sent two commanders, Ariyat and Abraha ibn al-Sabbah Abu Yaksum, with a large army. They entered Yemen, defeated Dhu Nuwas (who drowned in the sea), and Abyssinia took control of Yemen, ruled by these two commanders.

Arifat and Abraha fought over leadership, and eventually Abraha became the sole ruler of the Abyssinian army in Yemen. He wrote to the Negus, who admonished him, and Abraha sent gifts, including Yemeni soil in a bag. He boasted in his letter: “Step on this bag with your kingdom, and here is my head that I send to you.” The Negus was pleased and approved of his actions.

Abraha decided to build a magnificent church in Yemen, intending to divert the Arabs’ pilgrimage from the Kaaba to this church. It was so high and grand that it was called al-Qullays (“the lofty”) by the Arabs. However, the Arabs strongly rejected this, especially the Quraysh, who were angry at this challenge to their sacred site.

The March Toward the Kaaba

Abraha assembled a massive army to attack Mecca. He brought with him a huge elephant named Mahmoud, along with others (some accounts say 8 or 12 elephants). The plan was to use chains to demolish the Kaaba stone by stone. When the Arabs heard of this, they prepared to defend the Kaaba. A Yemeni noble, Dhu Nufar, gathered men to fight and protect the House of Allah.

Abdul Muttalib, the grandfather of the Prophet Muhammad, met Abraha when his army approached Mecca. He requested the return of his camels that had been taken, but he refused to negotiate the Kaaba itself, saying:

“This is the House of Allah, and if anyone prevents access to it, it is Allah Himself who protects it.”

Despite attempts to turn the elephants toward the Kaaba, Mahmoud refused to move. The army tried to force him in several directions, but the elephant would not advance.

Divine Intervention

Allah sent flocks of birds (Tayran Ababil) from the sea. Each bird carried three stones—one in its beak and two in its claws, the size of chickpeas or lentils. They dropped these stones on the Abyssinian army, destroying them. Many soldiers died instantly; some were killed gradually while fleeing. Abraha himself was wounded and died later in Yemen.

This event showed Allah’s protection over the Kaaba and foreshadowed the coming of the Prophet Muhammad (peace be upon him). It is mentioned in the Qur’an:

“Have you not seen how your Lord dealt with the people of the Elephant? Did He not make their plan go astray? And He sent against them birds in flocks, striking them with stones of baked clay, and made them like eaten straw.” (Surah Al-Fil 105:1–5)

The birds are called Ababil, meaning successive flocks or groups. Sijjil refers to hard stones mixed with clay. “Like eaten straw” indicates their complete destruction and humiliation.
"""

print(result[result['surah_no'] == 105])


      surah_no  ayah_start  ayah_end  \
1886       105           1         5   

                                                tafseer  
1886  \nThis [event] is one of the blessings Allah g...  


In [27]:
duplicates = result[result.duplicated(subset=['tafseer'], keep=False)]
duplicates

,surah_no,ayah_start,ayah_end,tafseer


In [26]:
result.to_csv(
    "tafsir_ibn_kathir_final.csv",
    index=False,
    encoding="utf-8-sig"
)